# Datamine PLOTVA Process Example

This notebook demonstrates how to configure and run the **`plotva`** process wrapper in `dmstudio`.

## Process Description

## PLOTVA

Generates a multi-value scatter plot file, where each data point is annotated by up to 10 fields centred on this point.

Individual character size, offsets, angle and decimal places must be specified interactively for each field.

The input file should contain just one entry for each location. If it contains more than one, then there will be multiple plots at that location.

### Entering Field Details

You need to enter the following information for each field to be plotted:

* Field Name,
* Character Size,
* X Offset,
* Y Offset,
* Angle,
* Number of Decimal Places.

Up to 10 fields can be specified terminating with a blank line or a comma in column 1. The data for each field must be specified on a single line separating data items with commas. When data entry is terminated the user is prompted to enter 0 to start plotting, 1 to add more fields or 2 to start data entry again.

### Input Files:

* **in** (Undefined):
  Input data file. This must contain X and Y fields and at least one value field.
  Required=Yes

* **proto** (Plot Prototype):
  Plot prototype file. Must contain the fields **X, Y, S1, S2** and **CODE** (numeric,
  explicit) and **XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE** (numeric, implicit). If these
  last 6 values set in **PROTO** , then corresponding parameters need not be set.
  Required=Yes

### Output Files:

* **plot** (Plot):
  Output plot file.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  Field to be plotted along X axis.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Field to be plotted along Y axis.
  Default=Undefined
  Required=Yes

* **symcode** (Numeric : IN):
  The **SYMCODE** value will control the symbol used on each point. If used, it will
  override the **SYMBOL** parameter.
  Default=Undefined
  Required=No

### Parameters:

* **angle**:
  Angle for symbol plotting in degrees from the X axis.(0)
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **symbol**:
  Plotted symbol at each point. Default (92). Point symbol number
  Range=
  Values=
  Default=
  Required=No

* **symsize**:
  Symbol size in millimetres (3). Set to 0 for no symbol.
  Range=Undefined
  Values=Undefined
  Default=3
  Required=No

* **aspratio**:
  Aspect ratio, width / ht. for chars (0.9).
  Range=Undefined
  Values=Undefined
  Default=0.9
  Required=No

* **append**:
  Plot append flag. If set to 1 then the new plot will be appended to the **PLOT** file,
  if it exists and is a valid plot file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **xmin**:
  Minimum value of X for plot. None of **XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE** need be
  set if this information is already in the prototype.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xmax**:
  Maximum value of X for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymin**:
  Minimum value of Y for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymax**:
  Maximum value of Y for plot.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xscale**:
  X scale in user data units per millimetre.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **yscale**:
  Y scale in user data units per millimetre.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('plotva')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute plotva
print("Running plotva...")
dm_cmd.plotva(
    in_i='t_assays',  # required
    proto_i='t_mod1',  # required
    plot_o='t_plotva_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    # symcode_f='optional',  # optional
    # angle_p=0,  # optional
    # symbol_p='optional',  # optional
    # symsize_p=3,  # optional
    # aspratio_p=0.9,  # optional
    # append_p=0,  # optional
    # xmin_p='optional',  # optional
    # xmax_p='optional',  # optional
    # ymin_p='optional',  # optional
    # ymax_p='optional',  # optional
    # xscale_p='optional',  # optional
    # yscale_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("plotva execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_plotva_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")